# DeBERTa-v3-large — Sarcasm Detector (Vast.ai Version · v2)
### Three accuracy improvements over v1

| Fix | Change | Expected gain |
|-----|--------|---------------|
| 1 | `MAX_LEN 128 → 256` | +1–2% |
| 2 | Layer-wise LR decay | +1–2% |
| 3 | `deberta-v3-base → deberta-v3-large` | +2–4% |

| Component | Value |
|-----------|-------|
| Model | `microsoft/deberta-v3-large` (400M params) |
| Dataset | `train-balanced-sarcasm.csv` (upload to `/workspace/data/`) |
| Checkpoints | saved to `/workspace/checkpoints/` |
| Expected accuracy | ~87–90% |

> ⚠️ **VRAM requirements:** deberta-v3-large needs ~20–22GB with batch_size=8.  
> RTX 3090 (24GB) ✅ · A40/A6000 (48GB) ✅ · RTX 3080 (10GB) ❌ use base instead.  
> Upload `train-balanced-sarcasm.csv` to `/workspace/data/` before running.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
import importlib
import subprocess

def ensure_installed(packages: dict):
    """
    Check if packages are installed, install only the missing ones.
    
    packages: dict of {import_name: pip_install_name}
    e.g. {'sklearn': 'scikit-learn', 'pandas': 'pandas'}
    """
    missing = []
    for import_name, pip_name in packages.items():
        if importlib.util.find_spec(import_name) is None:
            print(f'  ✗ {import_name} — not found, will install')
            missing.append(pip_name)
        else:
            print(f'  ✓ {import_name} — already installed')
    
    if missing:
        print(f'\nInstalling: {missing}')
        subprocess.run(['pip', 'install', '-q'] + missing, check=True)
        print('Done.')
    else:
        print('\nAll packages already installed.')

# Usage
ensure_installed({
    'pandas':       'pandas',
    'numpy':        'numpy',
    'torch':        'torch',
    'datasets':     'datasets',
    'sklearn':      'scikit-learn',
    'transformers': 'transformers==4.44.2',
    'sentencepiece':'sentencepiece==0.1.99',
    'accelerate':   'accelerate',
})

import subprocess
subprocess.run(['pip', 'uninstall', '-y', 'transformers', 'tokenizers', 'protobuf', 'sentencepiece', 'wandb'],
               capture_output=True)
subprocess.run(['pip', 'install', '-q',
                'transformers==4.44.2',
                'tokenizers==0.19.1',
                'sentencepiece==0.1.99',
                'protobuf==3.20.3',
                'datasets',
                'scikit-learn',
                'accelerate',
               'pandas',
               'numpy'],
               capture_output=True)

import os
os.environ['WANDB_DISABLED'] = 'true'
print('Installation complete.')

import subprocess, sys, os

# Install using the kernel's own python
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'datasets', 'scikit-learn',
                'transformers==4.44.2', 'tokenizers==0.19.1',
                'sentencepiece==0.1.99', 'protobuf==3.20.3',
                'accelerate'], check=True)

print('Done — restarting kernel now...')
os.kill(os.getpid(), 9)  # force kernel restart

  ✗ pandas — not found, will install
  ✓ numpy — already installed
  ✓ torch — already installed
  ✗ datasets — not found, will install
  ✗ sklearn — not found, will install
  ✗ transformers — not found, will install
  ✗ sentencepiece — not found, will install
  ✗ accelerate — not found, will install

Installing: ['pandas', 'datasets', 'scikit-learn', 'transformers==4.44.2', 'sentencepiece==0.1.99', 'accelerate']


Done.
Installation complete.


In [1]:
# ── Cell 2: Imports & GPU check ────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    DebertaV2Tokenizer,
    DebertaV2ForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
    if vram < 18:
        print('WARNING: < 18GB VRAM — consider switching back to deberta-v3-base')
else:
    print('WARNING: No GPU detected! Training will be very slow.')

PyTorch  : 2.11.0+cu128
Device   : cuda
GPU      : NVIDIA A100-PCIE-40GB
VRAM     : 42.4 GB


In [2]:
# ── Cell 3: Paths & config ─────────────────────────────────────────────────
DATA_PATH  = '/workspace/train-balanced-sarcasm.csv'
CKPT_DIR   = '/workspace/checkpoints/deberta-large-sarcasm'
FINAL_DIR  = '/workspace/checkpoints/deberta-large-sarcasm-final'

# ── FIX 3: deberta-v3-large instead of deberta-v3-base ────────────────────
MODEL_NAME = 'microsoft/deberta-v3-large'

# ── FIX 1: MAX_LEN 128 → 256 ──────────────────────────────────────────────
# Captures more context per Reddit comment — especially important when
# parent_comment is long. Doubles token budget at cost of ~2x memory.
MAX_LEN    = 256

os.makedirs('/workspace/data', exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}\n'
        'Please upload train-balanced-sarcasm.csv to /workspace/data/ '
        'using the Jupyter file browser on the left.'
    )
print(f'Model      : {MODEL_NAME}')
print(f'MAX_LEN    : {MAX_LEN}')
print(f'Dataset    : {DATA_PATH}')
print(f'Checkpoints: {CKPT_DIR}')

Model      : microsoft/deberta-v3-large
MAX_LEN    : 256
Dataset    : /workspace/train-balanced-sarcasm.csv
Checkpoints: /workspace/checkpoints/deberta-large-sarcasm


In [3]:
# ── Cell 4: Load & preprocess data ─────────────────────────────────────────
print('Loading dataset...')
df = pd.read_csv(DATA_PATH).dropna(subset=['comment'])

df['parent_comment'] = df['parent_comment'].fillna('')
df = df[['comment', 'parent_comment', 'label']]

# Comment first — the label applies to the comment, so put it in
# the primary (non-truncated) position. Parent gives context after [SEP].
df['text'] = df['comment'] + ' [SEP] ' + df['parent_comment']
df = df[['text', 'label']]

print(f'Total samples : {len(df):,}')
print(df['label'].value_counts())

Loading dataset...
Total samples : 1,010,771
label
0    505403
1    505368
Name: count, dtype: int64


In [4]:
# ── Cell 5: Train / val / test split ───────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}')

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

Train : 808,616  |  Val : 101,077  |  Test : 101,078


In [5]:
# ── Cell 6: Tokenizer ──────────────────────────────────────────────────────
print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size : {tokenizer.vocab_size:,}')

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,   # FIX 1: 256 instead of 128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)
print('Tokenization complete.')

Loading tokenizer: microsoft/deberta-v3-large


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

/venv/main/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Vocab size : 128,000


Map:   0%|          | 0/808616 [00:00<?, ? examples/s]

Map:   0%|          | 0/101077 [00:00<?, ? examples/s]

Map:   0%|          | 0/101078 [00:00<?, ? examples/s]

Tokenization complete.


In [6]:
# ── Cell 7: Load model ─────────────────────────────────────────────────────
# FIX 3: Loading deberta-v3-large (~400M params vs 183M for base)
print(f'Loading model: {MODEL_NAME}')
model = DebertaV2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable:,}')

Loading model: microsoft/deberta-v3-large


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters    : 435,063,810
Trainable parameters: 435,063,810


In [7]:
# ── Cell 8: Metrics ────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

In [12]:
# ── Cell 9: Training arguments ─────────────────────────────────────────────
# deberta-v3-large is ~2x the size of base — reduce batch size accordingly.
# gradient_accumulation_steps keeps the effective batch size at 32.
#
#   GPU VRAM     | per_device_train_batch_size | gradient_accumulation_steps
#   RTX 3090 24GB|  8                          |  4  → effective 32
#   A40/A6000 48G| 16                          |  2  → effective 32
#   A100 80GB    | 32                          |  1  → effective 32

TRAIN_BATCH   = 16
GRAD_ACCUM    = 2
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5

args = TrainingArguments(
    output_dir                  = CKPT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    per_device_eval_batch_size  = 16,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.15,
    weight_decay                = 0.01,
    evaluation_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'accuracy',
    greater_is_better           = True,
    logging_steps               = 200,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    dataloader_num_workers = 4,   # ← add this, CPU was bottlenecking GPU
    dataloader_pin_memory  = True, # ← faster CPU→GPU transfer
)

print('Training arguments set.')
print(f'  batch_size (per device)  : {args.per_device_train_batch_size}')
print(f'  gradient_accumulation    : {args.gradient_accumulation_steps}')
print(f'  effective batch_size     : {TRAIN_BATCH * GRAD_ACCUM}')
print(f'  learning_rate            : {args.learning_rate}')
print(f'  fp16                     : {args.fp16}')

Training arguments set.
  batch_size (per device)  : 16
  gradient_accumulation    : 2
  effective batch_size     : 32
  learning_rate            : 2e-05
  fp16                     : True


/venv/main/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [13]:
# ── Cell 10: FIX 2 — Layer-wise learning rate decay ────────────────────────
# Lower transformer layers learn general language features that change
# slowly; upper layers and the classifier head learn task-specific
# features and should update faster.
# We apply a decay factor of 0.9 per layer from top → bottom.
#
# deberta-v3-large has 24 hidden layers (vs 12 for base).

DECAY_RATE  = 0.9   # LR multiplier per layer going down

# Detect actual layer count from config
num_layers = model.config.num_hidden_layers
print(f'Number of hidden layers: {num_layers}')

# Build parameter groups: one per layer + embedding + classifier/pooler
no_decay = {'bias', 'LayerNorm.weight', 'LayerNorm.bias'}

optimizer_grouped_parameters = []

# Embeddings — lowest LR
embed_lr = LEARNING_RATE * (DECAY_RATE ** num_layers)
optimizer_grouped_parameters += [
    {
        'params': [p for n, p in model.named_parameters()
                   if 'embeddings' in n and not any(nd in n for nd in no_decay)],
        'lr': embed_lr, 'weight_decay': 0.01,
    },
    {
        'params': [p for n, p in model.named_parameters()
                   if 'embeddings' in n and any(nd in n for nd in no_decay)],
        'lr': embed_lr, 'weight_decay': 0.0,
    },
]

# One group per transformer layer
for layer_idx in range(num_layers):
    # Layer 0 (bottom) gets the smallest LR, layer N-1 (top) gets the largest
    layer_lr = LEARNING_RATE * (DECAY_RATE ** (num_layers - 1 - layer_idx))
    optimizer_grouped_parameters += [
        {
            'params': [p for n, p in model.named_parameters()
                       if f'layer.{layer_idx}.' in n
                       and not any(nd in n for nd in no_decay)],
            'lr': layer_lr, 'weight_decay': 0.01,
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if f'layer.{layer_idx}.' in n
                       and any(nd in n for nd in no_decay)],
            'lr': layer_lr, 'weight_decay': 0.0,
        },
    ]

# Classifier head & pooler — full LR
optimizer_grouped_parameters += [
    {
        'params': [p for n, p in model.named_parameters()
                   if ('classifier' in n or 'pooler' in n)
                   and not any(nd in n for nd in no_decay)],
        'lr': LEARNING_RATE, 'weight_decay': 0.01,
    },
    {
        'params': [p for n, p in model.named_parameters()
                   if ('classifier' in n or 'pooler' in n)
                   and any(nd in n for nd in no_decay)],
        'lr': LEARNING_RATE, 'weight_decay': 0.0,
    },
]

optimizer = AdamW(optimizer_grouped_parameters)

# Linear warmup + decay scheduler
num_training_steps = (
    len(train_dataset) // (TRAIN_BATCH * GRAD_ACCUM) + 1
) * NUM_EPOCHS
num_warmup_steps   = int(num_training_steps * 0.15)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = num_warmup_steps,
    num_training_steps = num_training_steps,
)

print(f'Layer-wise LR decay built.')
print(f'  Embedding LR : {embed_lr:.2e}')
print(f'  Top layer LR : {LEARNING_RATE:.2e}')
print(f'  Decay factor : {DECAY_RATE} per layer')
print(f'  Warmup steps : {num_warmup_steps} / {num_training_steps}')

Number of hidden layers: 24
Layer-wise LR decay built.
  Embedding LR : 1.60e-06
  Top layer LR : 2.00e-05
  Decay factor : 0.9 per layer
  Warmup steps : 18952 / 126350


In [14]:
# Check your actual effective throughput
# 1.24 it/s × batch 4 × accum 8 = 39.7 samples/sec
# Should be ~200+ samples/sec on a healthy 3090
print(f"Effective samples/sec: {1.24 * TRAIN_BATCH * GRAD_ACCUM:.0f}")
print(f"GPU utilization — run nvidia-smi in terminal to check")

Effective samples/sec: 40
GPU utilization — run nvidia-smi in terminal to check


In [15]:
# ── Cell 11: Train ─────────────────────────────────────────────────────────
# Pass the custom optimizer + scheduler via the optimizers argument.
# Trainer will skip its default AdamW and use ours with layer-wise LRs.

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    optimizers      = (optimizer, scheduler),   # FIX 2: layer-wise LR decay
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
0,0.416000,0.396284,0.821275,0.820911
2,0.288400,0.435525,0.830130,0.830129
4,0.173300,0.567851,0.826024,0.826014


TrainOutput(global_step=126345, training_loss=0.3025152166340083, metrics={'train_runtime': 62102.9386, 'train_samples_per_second': 65.103, 'train_steps_per_second': 2.034, 'total_flos': 1.8839156968964751e+18, 'train_loss': 0.3025152166340083, 'epoch': 4.999901066503097})

In [16]:
# ── Cell 12: Save final model ──────────────────────────────────────────────
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'Model + tokenizer saved to: {FINAL_DIR}')
print()
print('IMPORTANT: Download /workspace/checkpoints/ before stopping the instance!')
print('Terminal: tar -czf deberta_large_final.tar.gz /workspace/checkpoints/')

Model + tokenizer saved to: /workspace/checkpoints/deberta-large-sarcasm-final

IMPORTANT: Download /workspace/checkpoints/ before stopping the instance!
Terminal: tar -czf deberta_large_final.tar.gz /workspace/checkpoints/


In [17]:
# ── Cell 13: Evaluate on test set ──────────────────────────────────────────
predictions = trainer.predict(test_dataset)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(f'\nTest Accuracy : {accuracy_score(labels, preds):.4f}')
print(f'Test F1 Score : {f1_score(labels, preds, average="weighted"):.4f}')
print('\nFull Classification Report:')
print(classification_report(labels, preds, target_names=['not sarcastic', 'sarcastic']))


Test Accuracy : 0.8306
Test F1 Score : 0.8306

Full Classification Report:
               precision    recall  f1-score   support

not sarcastic       0.83      0.83      0.83     50541
    sarcastic       0.83      0.83      0.83     50537

     accuracy                           0.83    101078
    macro avg       0.83      0.83      0.83    101078
 weighted avg       0.83      0.83      0.83    101078



In [18]:
# ── Cell 14: Package model for download ────────────────────────────────────
import subprocess
result = subprocess.run(
    ['tar', '-czf', '/workspace/deberta_large_sarcasm_final.tar.gz',
     '-C', '/workspace/checkpoints', '.'],
    capture_output=True, text=True
)
size = os.path.getsize('/workspace/deberta_large_sarcasm_final.tar.gz') / 1e6
print(f'Archive: /workspace/deberta_large_sarcasm_final.tar.gz  ({size:.0f} MB)')
print('Download from the Jupyter file browser (left panel).')

Archive: /workspace/deberta_large_sarcasm_final.tar.gz  (23752 MB)
Download from the Jupyter file browser (left panel).
